<a href="https://colab.research.google.com/github/DeepthiManthapuram/Deep_Learning/blob/main/RNN_LSTM_GRU_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

DATASET : from tensorflow.keras.datasets import imdb

Problem Statement
A movie streaming company wants to automatically analyze customer reviews and determine whether a review is:
Positive
Negative
The company currently receives thousands of reviews daily and manual analysis is impossible.
As an AI Engineer, your task is to build and compare three Deep Learning models:
SimpleRNN
LSTM
GRU
Then identify which architecture performs best for sentiment analysis.

Project Objectives
Build an end-to-end NLP application that:
processes movie reviews
trains three sequence models
compares performance
deploys the best model using Streamlit

Task 1: Dataset Analysis
Perform exploratory analysis:
Number of reviews
Positive reviews count
Negative reviews count
Average review length
Longest review
Shortest review

In [ ]:
# Task 1: Dataset Analysis - IMDB Movie Reviews Dataset

from tensorflow.keras.datasets import imdb
import pandas as pd
import numpy as np

# Load dataset
(X_train, y_train), (X_test, y_test) = imdb.load_data()

# Combine train and test sets
reviews = list(X_train) + list(X_test)
labels = np.concatenate([y_train, y_test])

# Total reviews
total_reviews = len(reviews)

# Positive and Negative reviews
positive_reviews = np.sum(labels == 1)
negative_reviews = np.sum(labels == 0)

# Review lengths
review_lengths = [len(review) for review in reviews]

avg_length = np.mean(review_lengths)
longest_review = np.max(review_lengths)
shortest_review = np.min(review_lengths)

# Create summary dataframe
summary = pd.DataFrame({
    "Metric": [
        "Total Reviews",
        "Positive Reviews",
        "Negative Reviews",
        "Average Review Length",
        "Longest Review Length",
        "Shortest Review Length"
    ],
    "Value": [
        total_reviews,
        positive_reviews,
        negative_reviews,
        round(avg_length, 2),
        longest_review,
        shortest_review
    ]
})

print("\nIMDB Dataset Analysis")
print("=" * 40)
print(summary)

# Additional Statistics
print("\nReview Length Statistics")
print("=" * 40)
print(f"Mean Length   : {np.mean(review_lengths):.2f}")
print(f"Median Length : {np.median(review_lengths):.2f}")
print(f"Std Dev       : {np.std(review_lengths):.2f}")

# Find longest and shortest review indices
longest_idx = np.argmax(review_lengths)
shortest_idx = np.argmin(review_lengths)

print("\nLongest Review Length :", review_lengths[longest_idx])
print("Shortest Review Length:", review_lengths[shortest_idx])


IMDB Dataset Analysis
                   Metric     Value
0           Total Reviews  50000.00
1        Positive Reviews  25000.00
2        Negative Reviews  25000.00
3   Average Review Length    234.76
4   Longest Review Length   2494.00
5  Shortest Review Length      7.00

Review Length Statistics
Mean Length   : 234.76
Median Length : 176.00
Std Dev       : 172.91

Longest Review Length : 2494
Shortest Review Length: 7


Task 2: Text Preprocessing
Apply:
Lowercase conversion
HTML tag removal
Punctuation removal
Stopword removal
Tokenization
Sequence generation
Padding

In [ ]:
# Task 2: Text Preprocessing

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import re
import string
from nltk.corpus import stopwords
import nltk

# Download stopwords (run once)
nltk.download('stopwords')

# Load IMDB dataset - IMPORTANT: Load the raw data first, we will preprocess it
VOCAB_SIZE = 10000 # Use a consistent VOCAB_SIZE for loading
(X_train_raw, y_train), (X_test_raw, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

# Get word index for decoding (needed to convert raw integer sequences back to text)
word_index = imdb.get_word_index()
# The first 3 indices are reserved for padding, start-of-sequence, and unknown
reverse_word_index = {value + 3: key for key, value in word_index.items()}
reverse_word_index[0] = "<pad>"
reverse_word_index[1] = "<sos>"
reverse_word_index[2] = "<unk>"

# Function to decode reviews (now applies to all raw data)
def decode_review(encoded_review):
    return ' '.join(
        [reverse_word_index.get(i, '?') for i in encoded_review] # Adjusted to use the 0-indexed word_index map correctly
    )

# Convert all encoded reviews to text
train_reviews_text = [decode_review(review) for review in X_train_raw]
test_reviews_text = [decode_review(review) for review in X_test_raw]

# Stopwords
stop_words = set(stopwords.words('english'))

# Preprocessing function (remains the same)
def preprocess_text(text):
    # Lowercase
    text = text.lower()
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    # Tokenization
    tokens = text.split()
    # Stopword removal
    tokens = [word for word in tokens if word not in stop_words]
    return " ".join(tokens)

# Apply preprocessing to all training and testing reviews
clean_train_reviews = [preprocess_text(review) for review in train_reviews_text]
clean_test_reviews = [preprocess_text(review) for review in test_reviews_text]

print("Original Training Review (first 300 chars):")
print(train_reviews_text[0][:300]) # Use original decoded text
print("\nCleaned Training Review (first 300 chars):")
print(clean_train_reviews[0][:300])

# Tokenization - fit on training data only
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(clean_train_reviews)

# Sequence generation for both training and testing sets
train_sequences = tokenizer.texts_to_sequences(clean_train_reviews)
test_sequences = tokenizer.texts_to_sequences(clean_test_reviews)

print("\nSample Training Sequence (first 20 tokens):")
print(train_sequences[0][:20])

# Padding for both training and testing sets
MAX_LEN = 200

# Overwrite X_train and X_test with the padded preprocessed data
X_train = pad_sequences(
    train_sequences,
    maxlen=MAX_LEN,
    padding='post',
    truncating='post'
)
X_test = pad_sequences(
    test_sequences,
    maxlen=MAX_LEN,
    padding='post',
    truncating='post'
)

print("\nPadded Training Shape:")
print(X_train.shape)
print("\nPadded Test Shape:")
print(X_test.shape)

print("\nFirst Padded Training Review:")
print(X_train[0])


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step
Original Training Review (first 300 chars):
<sos> this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert <unk> is an amazing actor and now the same being director <unk> father came from the same scottish island as myself so i loved the f

Cleaned Training Review (first 300 chars):
film brilliant casting location scenery story direction everyones really suited part played could imagine robert amazing actor director father came scottish island loved fact real connection film witty remarks throughout film great brilliant much bought film soon released would recommend everyone wa

Sample Training Sequence (first 20 tokens):
[4, 414, 838, 1472, 1239, 11, 343, 4253, 12, 3742, 77, 153, 28, 710, 548, 364, 178, 73, 226, 273]

Padded Training Shape:
(25000, 200)

Padded Test Shape:
(25000, 20

In [ ]:
import time
# from tensorflow.keras.datasets import imdb # Removed, data loaded in Task 2
# from tensorflow.keras.preprocessing.sequence import pad_sequences # Removed, data padded in Task 2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.callbacks import EarlyStopping

# VOCAB_SIZE and MAX_LEN are now set in the preprocessing cell (x4NPzvbuvqRz)
# and X_train, y_train, X_test, y_test are also globally available from there.
# We define them here again for clarity and self-containment if running just this cell.
VOCAB_SIZE = 10000
MAX_LEN = 200

# The data loading and padding is handled in Task 2 (cell x4NPzvbuvqRz).
# (X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE) # REMOVED
# X_train = pad_sequences(X_train, maxlen=MAX_LEN, padding='post') # REMOVED
# X_test = pad_sequences(X_test, maxlen=MAX_LEN, padding='post') # REMOVED

print("Training Shape:", X_train.shape)
print("Testing Shape :", X_test.shape)

# Build SimpleRNN Model
rnn_model = Sequential([
    Embedding(input_dim=VOCAB_SIZE,
              output_dim=128),

    SimpleRNN(64),

    Dense(32, activation='relu'),

    Dense(1, activation='sigmoid')
])

# Model Summary
rnn_model.summary()

# Compile Model
rnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Early Stopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

# Train Model
start_time = time.time()
rnn_history = rnn_model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stop]
)
rnn_training_time = time.time() - start_time

# Evaluate Model
rnn_loss, rnn_accuracy = rnn_model.evaluate(X_test, y_test)

print("\nTest Loss:", round(rnn_loss, 4))
print("Test Accuracy:", round(rnn_accuracy * 100, 2), "%")

# Save Model
rnn_model.save("imdb_simplernn_model.h5")
print("\nSimpleRNN model saved successfully!")


Training Shape: (25000, 200)
Testing Shape : (25000, 200)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 10s 32ms/step - accuracy: 0.5026 - loss: 0.6953 - val_accuracy: 0.5054 - val_loss: 0.6942
Epoch 2/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.5066 - loss: 0.6935 - val_accuracy: 0.4998 - val_loss: 0.6989
Epoch 3/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.5213 - loss: 0.6895 - val_accuracy: 0.5034 - val_loss: 0.6978
782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.5090 - loss: 0.6938



Test Loss: 0.6938
Test Accuracy: 50.9 %

SimpleRNN model saved successfully!


In [ ]:
import time
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.models import Sequential # Ensure Sequential is imported
from tensorflow.keras.callbacks import EarlyStopping # Ensure EarlyStopping is imported

# VOCAB_SIZE and MAX_LEN are now globally available from the preprocessing cell (x4NPzvbuvqRz)
# So, remove local definitions to avoid confusion and ensure consistency.

# Build LSTM Model
lstm_model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, # VOCAB_SIZE is global from x4NPzvbuvqRz
              output_dim=128),

    LSTM(64),

    Dense(32, activation='relu'),

    Dense(1, activation='sigmoid')
])

# Model Summary
lstm_model.summary()

# Compile Model
lstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Early Stopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

# Train Model
start_time = time.time()
lstm_history = lstm_model.fit(
    X_train, # X_train, y_train are global from x4NPzvbuvqRz
    y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stop]
)
lstm_training_time = time.time() - start_time

# Evaluate Model
lstm_loss, lstm_accuracy = lstm_model.evaluate(X_test, y_test) # X_test, y_test are global from x4NPzvbuvqRz

print("\nTest Loss:", round(lstm_loss, 4))
print("Test Accuracy:", round(lstm_accuracy * 100, 2), "%")

# Save Model
lstm_model.save("imdb_lstm_model.h5")
print("\nLSTM model saved successfully!")


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 7s 21ms/step - accuracy: 0.5058 - loss: 0.6946 - val_accuracy: 0.5020 - val_loss: 0.6941
Epoch 2/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.5240 - loss: 0.6871 - val_accuracy: 0.5216 - val_loss: 0.6879
Epoch 3/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.5419 - loss: 0.6551 - val_accuracy: 0.5358 - val_loss: 0.6807
Epoch 4/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.5667 - loss: 0.6221 - val_accuracy: 0.6066 - val_loss: 0.6883
Epoch 5/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.8349 - loss: 0.4015 - val_accuracy: 0.8478 - val_loss: 0.3770
Epoch 6/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.9171 - loss: 0.2314 - val_accuracy: 0.8578 - val_loss: 0.3668
Epoch 7/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.9558 - loss: 0.1433 - val_accuracy: 0.8586 - val_loss: 0.4333
Epoch 8/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.9754 - loss: 0.0857 - val_accu


Test Loss: 0.3649
Test Accuracy: 85.74 %

LSTM model saved successfully!


In [ ]:
import time
from tensorflow.keras.layers import Embedding, GRU, Dense
from tensorflow.keras.models import Sequential # Ensure Sequential is imported
from tensorflow.keras.callbacks import EarlyStopping # Ensure EarlyStopping is imported

# VOCAB_SIZE and MAX_LEN are now globally available from the preprocessing cell (x4NPzvbuvqRz)
# So, remove local definitions to avoid confusion and ensure consistency.

# Build GRU Model
gru_model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, # VOCAB_SIZE is global from x4NPzvbuvqRz
              output_dim=128),

    GRU(64),

    Dense(32, activation='relu'),

    Dense(1, activation='sigmoid')
])

# Model Summary
gru_model.summary()

# Compile Model
gru_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Early Stopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

# Train Model
start_time = time.time()
gru_history = gru_model.fit(
    X_train, # X_train, y_train are global from x4NPzvbuvqRz
    y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stop]
)
gru_training_time = time.time() - start_time

# Evaluate Model
gru_loss, gru_accuracy = gru_model.evaluate(X_test, y_test) # X_test, y_test are global from x4NPzvbuvqRz

print("\nTest Loss:", round(gru_loss, 4))
print("Test Accuracy:", round(gru_accuracy * 100, 2), "%")

# Save Model
gru_model.save("imdb_gru_model.h5")
print("\nGRU model saved successfully!")


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.5064 - loss: 0.6931 - val_accuracy: 0.5042 - val_loss: 0.6938
Epoch 2/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.5341 - loss: 0.6780 - val_accuracy: 0.5100 - val_loss: 0.6953
Epoch 3/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.5594 - loss: 0.6330 - val_accuracy: 0.5158 - val_loss: 0.7269
782/782 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.5053 - loss: 0.6926



Test Loss: 0.6926
Test Accuracy: 50.53 %

GRU model saved successfully!


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ------------------------------
# Store metrics for each model
# ------------------------------

results = []

# ===== SimpleRNN =====
y_pred_rnn = (rnn_model.predict(X_test) > 0.5).astype("int32")

results.append({
    "Model": "SimpleRNN",
    "Accuracy": accuracy_score(y_test, y_pred_rnn),
    "Precision": precision_score(y_test, y_pred_rnn),
    "Recall": recall_score(y_test, y_pred_rnn),
    "F1 Score": f1_score(y_test, y_pred_rnn),
    "Training Time (sec)": round(rnn_training_time, 2),
    "Validation Accuracy": max(rnn_history.history['val_accuracy']),
    "Validation Loss": min(rnn_history.history['val_loss']),
    "Parameters": rnn_model.count_params()
})

# ===== LSTM =====
y_pred_lstm = (lstm_model.predict(X_test) > 0.5).astype("int32")

results.append({
    "Model": "LSTM",
    "Accuracy": accuracy_score(y_test, y_pred_lstm),
    "Precision": precision_score(y_test, y_pred_lstm),
    "Recall": recall_score(y_test, y_pred_lstm),
    "F1 Score": f1_score(y_test, y_pred_lstm),
    "Training Time (sec)": round(lstm_training_time, 2),
    "Validation Accuracy": max(lstm_history.history['val_accuracy']),
    "Validation Loss": min(lstm_history.history['val_loss']),
    "Parameters": lstm_model.count_params()
})

# ===== GRU =====
y_pred_gru = (gru_model.predict(X_test) > 0.5).astype("int32")

results.append({
    "Model": "GRU",
    "Accuracy": accuracy_score(y_test, y_pred_gru),
    "Precision": precision_score(y_test, y_pred_gru),
    "Recall": recall_score(y_test, y_pred_gru),
    "F1 Score": f1_score(y_test, y_pred_gru),
    "Training Time (sec)": round(gru_training_time, 2),
    "Validation Accuracy": max(gru_history.history['val_accuracy']),
    "Validation Loss": min(gru_history.history['val_loss']),
    "Parameters": gru_model.count_params()
})

# Create comparison table
comparison_df = pd.DataFrame(results)

# Round values
comparison_df.iloc[:, 1:7] = comparison_df.iloc[:, 1:7].round(4)

print("\nMODEL PERFORMANCE COMPARISON")
print("=" * 100)
print(comparison_df)

# Save comparison table
comparison_df.to_csv("model_comparison.csv", index=False)

782/782 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step
782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step

MODEL PERFORMANCE COMPARISON
       Model  Accuracy  Precision  Recall  F1 Score  Training Time (sec)  \
0  SimpleRNN    0.5509     0.5362  0.7541    0.6267                18.09   
1       LSTM    0.7821     0.8287  0.7112    0.7655                16.05   
2        GRU    0.8651     0.8709  0.8574    0.8641                18.07   

   Validation Accuracy  Validation Loss  Parameters  
0               0.5576         0.671969     1294465  
1               0.7902         0.488603     1331521  
2               0.8720         0.326683     1319361  
